In [ ]:
import pandas as pd
import numpy as np

**<h2 align ="center" style='color: white'>Model which Classifies customers based based on risk</h2>**

In [ ]:
cust_data_mixed2 = pd.read_csv('data/CustDataMixed2.csv')
cust_data_mixed2.head()

In [ ]:
cust_data_mixed2.shape

In [ ]:
cust_data_mixed2.head()

In [ ]:
# random_columns2 = cust_data_mixed2.sample(n=32240, axis=0, random_state=42,replace=True)
# random_columns2.head(2)

**<h4 align ="left" style='color: white'>Working with sample size of 32,240</h4>**

In [ ]:
random_columns2 = cust_data_mixed2.sample(n=32240, axis=0, random_state=42,replace=True)
random_columns2.head()

In [ ]:
random_columns2.shape

In [ ]:
# give me a new dataset of 1200 columns from cust_data_mixed2

In [ ]:
df = random_columns2.copy()

In [ ]:
df.head(5)

In [ ]:
y = df['Risk Level']
X = df[['Age', 'Status', 'city','zip']]

In [ ]:
y.head(5)

In [ ]:
X.head(5)

**<h4 align ="left" style='color: white'>One hot encoding for categorical variables in X which is norminal (Variables that are not ranked)</h4>**

In [ ]:
X_encoded = pd.get_dummies(X, columns=['Status', 'city'], drop_first=True)
X_encoded.head()

**<h4 align ="left" style='color: white'>OrdinalEncoder encoding for y which is Ordinal Variable  (Variable which can be ranked) </h4>**

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Reshape y to a 2D array as required by OrdinalEncoder
y_reshaped = y.values.reshape(-1, 1)

# Apply OrdinalEncoder
ordinal_encoder = OrdinalEncoder()
y_encoded = ordinal_encoder.fit_transform(y_reshaped)

# Convert back to a pandas Series for convenience
y_encoded = pd.Series(y_encoded.flatten(), name='Risk Level Encoded')

y_encoded.head()

In [ ]:
from sklearn.model_selection import train_test_split

# Perform train-test split with stratification to handle class imbalance
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Display the shapes of the resulting datasets
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# Calculate percentages
total_samples = len(y)
train_percentage = (len(y_train) / total_samples) * 100
test_percentage = (len(y_test) / total_samples) * 100

# Print the results
print(f"Train dataset percentage: {train_percentage:.2f}%")
print(f"Test dataset percentage: {test_percentage:.2f}%")

**<h4 align ="left" style='color: white'>Training and comparing different models </h4>**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report

# Define the models to compare
models = {
    "Logistic Regression": LogisticRegression(max_iter=500, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Support Vector Machine": SVC(random_state=42),
    "xgb_model": XGBClassifier(random_state=42),
    "knn_model" : KNeighborsClassifier(n_neighbors=5),
    "ada_model " : AdaBoostClassifier(random_state=42),
    "mlp_model" : MLPClassifier(random_state=42, max_iter=500),
    "svm_model" : SVC(class_weight='balanced', random_state=42)

   
    
    
}

# Dictionary to store results
results = {}

# Train and evaluate each model
for model_name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)
    
    # Predict on the test set
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Store the results
    results[model_name] = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    }
    
    # Print classification report
    print(f"Model: {model_name}")
    print(classification_report(y_test, y_pred))
    print("-" * 50)

# Display the results
for model_name, metrics in results.items():
    print(f"{model_name}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")
    print()

**<h4 align ="left" style='color: white'>Model evaluation with Confusion Matrix</h4>**

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

import matplotlib.pyplot as plt

# Plot confusion matrices for each model
fig, axes = plt.subplots(nrows=(len(models) + 2) // 2, ncols=2, figsize=(20, 20))
axes = axes.flatten()

for idx, (model_name, model) in enumerate(models.items()):
    # Predict on the test set
    y_pred = model.predict(X_test)
    
    # Plot confusion matrix
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[idx], display_labels=ordinal_encoder.categories_[0])
    axes[idx].set_title(f"Confusion Matrix: {model_name}")

# Hide any unused subplots
for i in range(len(models), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

**<h4 align ="left" style='color: white'>Results Based on F1, Precision and Accuracy</h4>**

In [ ]:
# Function to rank models based on a specific metric
def rank_models(results, metric):
    return sorted(results.items(), key=lambda x: x[1][metric], reverse=True)

# Rank models based on F1 Score, Precision, and Accuracy
ranked_by_f1 = rank_models(results, "F1 Score")
ranked_by_precision = rank_models(results, "Precision")
ranked_by_accuracy = rank_models(results, "Accuracy")

# Display rankings
print("Ranking by F1 Score:")
for rank, (model, scores) in enumerate(ranked_by_f1, start=1):
    print(f"{rank}. {model}: {scores['F1 Score']:.4f}")

print("\nRanking by Precision:")
for rank, (model, scores) in enumerate(ranked_by_precision, start=1):
    print(f"{rank}. {model}: {scores['Precision']:.4f}")

print("\nRanking by Accuracy:")
for rank, (model, scores) in enumerate(ranked_by_accuracy, start=1):
    print(f"{rank}. {model}: {scores['Accuracy']:.4f}")

In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# from lightgbm import LGBMClassifier
# # from catboost import CatBoostClassifier
# from sklearn.svm import SVC
# from sklearn.neural_network import MLPClassifier
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.ensemble import AdaBoostClassifier
# from sklearn.tree import DecisionTreeClassifier
# from xgboost import XGBClassifier



# rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)
# rf_model.fit(X_train, y_train)
# y_pred_rf = rf_model.predict(X_test)
   

# # lgbm_model = LGBMClassifier(class_weight='balanced', random_state=42)
# # lgbm_model.fit(X_train, y_train)
# # y_pred_lgbm = lgbm_model.predict(X_test)
   

# svm_model = SVC(class_weight='balanced', random_state=42)
# svm_model.fit(X_train, y_train)
# y_pred_svm = svm_model.predict(X_test)
   

# mlp_model = MLPClassifier(random_state=42, max_iter=500)
# mlp_model.fit(X_train, y_train)
# y_pred_mlp = mlp_model.predict(X_test)
   

# knn_model = KNeighborsClassifier(n_neighbors=5)
# knn_model.fit(X_train, y_train)
# y_pred_knn = knn_model.predict(X_test)
    

# ada_model = AdaBoostClassifier(random_state=42)
# ada_model.fit(X_train, y_train)
# y_pred_ada = ada_model.predict(X_test)


# dec_tree_Model = DecisionTreeClassifier(random_state=42),
# dec_tree_Model.fit(X_train, y_train)
# y_pred_decTree = dec_tree_Model.predict(X_test)


# # xgboost_model = XGBClassifier(random_state=42)
# # xgboost_model.fit(X_train, y_train)
# # y_pred_xgboost = xgboost_model.predict(X_test)
    

In [ ]:
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# import matplotlib.pyplot as plt

# # Define a function to plot confusion matrices
# def plot_confusion_matrices(y_test, predictions, model_names):
#     fig, axes = plt.subplots(2, 3, figsize=(15, 10))  # Create a 2x3 grid for 6 models
#     axes = axes.flatten()
    
#     for i, (y_pred, model_name) in enumerate(zip(predictions, model_names)):
#         cm = confusion_matrix(y_test, y_pred)
#         disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=ordinal_encoder.categories_[0])
#         disp.plot(ax=axes[i], cmap='Blues', colorbar=False)
#         axes[i].set_title(model_name)
    
#     plt.tight_layout()
#     plt.show()

# # Predictions and model names
# predictions = [y_pred_rf, y_pred_svm, y_pred_mlp, y_pred_knn, y_pred_ada]
# model_names = ['Random Forest', 'SVM', 'MLP', 'KNN', 'AdaBoost']

# # Plot the confusion matrices
# plot_confusion_matrices(y_test, predictions, model_names)

In [ ]:
# from sklearn.metrics import precision_score

# # Calculate precision, accuracy, and F1 score for each model
# results = []
# for model_name, y_pred in zip(model_names, predictions):
#     precision = precision_score(y_test, y_pred, average='weighted')
#     accuracy = accuracy_score(y_test, y_pred)
#     f1 = f1_score(y_test, y_pred, average='weighted')
#     results.append({'Model': model_name, 'Precision': precision, 'Accuracy': accuracy, 'F1 Score': f1})

# # Create a DataFrame to display the results
# performance_df = pd.DataFrame(results)
# print(performance_df)

In [ ]:
# # Rank models based on Precision, Accuracy, and F1 Score
# precision_rank = performance_df.sort_values(by='Precision', ascending=False)
# accuracy_rank = performance_df.sort_values(by='Accuracy', ascending=False)
# f1_score_rank = performance_df.sort_values(by='F1 Score', ascending=False)

# # Display the rankings
# print("Ranking based on Precision:")
# print(precision_rank)

# print("\nRanking based on Accuracy:")
# print(accuracy_rank)

# print("\nRanking based on F1 Score:")
# print(f1_score_rank)